# Bonus — Label-Efficiency Ablation Study (Consolidated)

**Group J** — CSE 445 Part B Bonus Task

This notebook consolidates the results from the seven individual per-run bonus notebooks
(`groupj-partb-bonus-coco-rho0-10/40/50`, `groupj-partb-bonus-dinov3-rho10/30/40/50`) plus the
shared 20%-label point reused from the main Task 1 pipeline (Table 4 of the report), into the
single results table, headline curve, and break-even analysis required by the assignment
specification (Section 6.2).

## ⚠️ Known data-consistency issue (carried over from the report, unresolved)

The report's own Section 5.4 flags this explicitly: **COCO's mAP50-95 at 10% labels (0.431)
is more than double COCO's mAP50-95 at 20% labels (0.2073)** — and higher than the 100%-label
Part A reference itself (0.333). An increase in label budget should not cause a large mAP
*decrease*. Suspected causes (per the report): a leakage bug in the nested label-fraction subset
builder, evaluation against a different split than the shared test partition, or a genuine
configuration mismatch (the ablation notebooks report SGD, lr=0.01; the main pipeline used
Adam, lr0=1e-3 — this should be confirmed against each run's actual `args.yaml`).

**This notebook reports the numbers as they currently stand, for consolidation purposes only.**
Treat the resulting curve and break-even analysis as illustrative, not verified, until the
subset-overlap check and optimiser/LR settings are confirmed.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Consolidated results table.
# Sources:
#   - 20% row: reused directly from Table 4 of the report (main Task 1 pipeline, shared for both initialisations)
#   - 10/40/50% COCO rows: groupj-partb-bonus-coco-rho0-{10,40,50}.ipynb
#   - 10/30/40/50% DINOv3 rows: groupj-partb-bonus-dinov3-rho{10,30,40,50}.ipynb
#   - 30% COCO: MISSING — groupj-partb-bonus-coco-rho0-30.ipynb was never run (see note below)
data = [
    {'label_fraction': 0.10, 'init': 'COCO',   'mAP50': 0.487,  'mAP50_95': 0.431,  'source_notebook': 'groupj-partb-bonus-coco-rho0-10.ipynb'},
    {'label_fraction': 0.10, 'init': 'DINOv3', 'mAP50': 0.491,  'mAP50_95': 0.419,  'source_notebook': 'groupj-partb-bonus-dinov3-rho10.ipynb'},
    {'label_fraction': 0.20, 'init': 'COCO',   'mAP50': 0.4796, 'mAP50_95': 0.2073, 'source_notebook': 'reused from Table 4 (Task 1 main pipeline)'},
    {'label_fraction': 0.20, 'init': 'DINOv3', 'mAP50': 0.4679, 'mAP50_95': 0.2054, 'source_notebook': 'reused from Table 4 (Task 1 main pipeline)'},
    {'label_fraction': 0.30, 'init': 'COCO',   'mAP50': np.nan, 'mAP50_95': np.nan, 'source_notebook': 'MISSING — groupj-partb-bonus-coco-rho0-30.ipynb not run'},
    {'label_fraction': 0.30, 'init': 'DINOv3', 'mAP50': 0.525,  'mAP50_95': 0.483,  'source_notebook': 'groupj-partb-bonus-dinov3-rho30.ipynb'},
    {'label_fraction': 0.40, 'init': 'COCO',   'mAP50': 0.555,  'mAP50_95': 0.520,  'source_notebook': 'groupj-partb-bonus-coco-rho0-40.ipynb'},
    {'label_fraction': 0.40, 'init': 'DINOv3', 'mAP50': 0.541,  'mAP50_95': 0.498,  'source_notebook': 'groupj-partb-bonus-dinov3-rho40.ipynb'},
    {'label_fraction': 0.50, 'init': 'COCO',   'mAP50': 0.555,  'mAP50_95': 0.520,  'source_notebook': 'groupj-partb-bonus-coco-rho0-50.ipynb'},
    {'label_fraction': 0.50, 'init': 'DINOv3', 'mAP50': 0.550,  'mAP50_95': 0.515,  'source_notebook': 'groupj-partb-bonus-dinov3-rho50.ipynb'},
]

PART_A_REFERENCE_MAP50_95 = 0.333  # Part A, 100% labels, optimistic per report Table 4 footnote

results_df = pd.DataFrame(data)
print("=== Consolidated Label-Efficiency Results ===")
print(results_df.to_string(index=False))
print()
print("NOTE: precision, recall, and training time per run are not consolidated here —")
print("the seven per-run notebooks (see 'source_notebook' column) only logged mAP50/mAP50-95")
print("for this bonus study. Pull precision/recall/training-time from each run notebook directly")
print("if needed for the report appendix.")


## Headline figure: label-efficiency curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

for init, color, marker in [('COCO', 'tab:blue', 'o'), ('DINOv3', 'tab:orange', 's')]:
    sub = results_df[results_df['init'] == init].sort_values('label_fraction')
    ax.plot(sub['label_fraction'] * 100, sub['mAP50_95'], marker=marker, color=color,
            label=f'{init}-initialised', linewidth=2, markersize=8)

ax.axhline(PART_A_REFERENCE_MAP50_95, color='gray', linestyle='--', linewidth=1.5,
           label=f'Part A reference, 100% labels ({PART_A_REFERENCE_MAP50_95:.3f}, optimistic)')

ax.set_xlabel('Label fraction (%)', fontsize=12)
ax.set_ylabel('Test mAP50-95', fontsize=12)
ax.set_title('Label-Efficiency Curve: Detection Performance vs. Label Budget', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/label_efficiency_curve.png', dpi=150)
plt.show()

print("\nNOTE: the 10% COCO point (0.431) breaks monotonicity with the 20% point (0.2073) —")
print("visible as a sharp drop-then-recover in the COCO line above. This is the unresolved")
print("data-consistency issue flagged at the top of this notebook, not a real finding.")


## Break-even analysis

In [ ]:
print("=== Break-even analysis ===\n")

# (a) Smallest rho where SSL-initialised (DINOv3) matches/exceeds the supervised (COCO) baseline at 50% labels
coco_50 = results_df.query("init=='COCO' and label_fraction==0.50")['mAP50_95'].values[0]
dinov3_sorted = results_df[results_df['init']=='DINOv3'].sort_values('label_fraction')
match_50 = dinov3_sorted[dinov3_sorted['mAP50_95'] >= coco_50]
if len(match_50):
    rho_a = match_50.iloc[0]['label_fraction']
    print(f"(a) DINOv3 first matches/exceeds COCO@50% (mAP50-95={coco_50:.3f}) at rho = {rho_a:.0%}")
else:
    print(f"(a) DINOv3 never matches COCO@50% (mAP50-95={coco_50:.3f}) within the tested range 10-50%")

# (b) Smallest rho where SSL-initialised (DINOv3) reaches 95% of the Part A 100%-label reference
target_95pct = 0.95 * PART_A_REFERENCE_MAP50_95
match_95 = dinov3_sorted[dinov3_sorted['mAP50_95'] >= target_95pct]
if len(match_95):
    rho_b = match_95.iloc[0]['label_fraction']
    print(f"(b) DINOv3 first reaches 95% of the Part A reference (target={target_95pct:.3f}) at rho = {rho_b:.0%}")
else:
    print(f"(b) DINOv3 never reaches 95% of the Part A reference (target={target_95pct:.3f}) within 10-50% labels")
    print(f"    Highest DINOv3 mAP50-95 achieved: {dinov3_sorted['mAP50_95'].max():.3f} at rho={dinov3_sorted.loc[dinov3_sorted['mAP50_95'].idxmax(),'label_fraction']:.0%}")

print()
print("Caveat: this break-even analysis inherits the unresolved COCO-10% data-consistency issue.")
print("The COCO comparison points above rho=10% (i.e. 40%, 50%) are internally consistent with each")
print("other and with Table 4's 20% point, so break-even conclusions using rho >= 20% are more trustworthy")
print("than any conclusion resting on the COCO 10% data point specifically.")


## Missing pieces for full spec compliance

Two required outputs (Section 6.2) are **not** included in this consolidation because the
underlying data doesn't exist yet:

1. **`coco-rho0-30` data point** — the notebook for this run (`groupj-partb-bonus-coco-rho0-30.ipynb`)
   was never created/run. Without it, the COCO line's 30% point is missing entirely (shown as a gap
   in the table and curve above). This should be run before the ablation study is considered complete.
2. **Qualitative panel** (same 3 test images at rho=0.1 vs rho=0.5, side by side) — this requires
   pulling actual prediction images from the individual `groupj-partb-bonus-*.ipynb` run notebooks,
   which this consolidation notebook doesn't have direct access to. Copy the relevant output cells
   from those notebooks in manually, or re-run inference here if the model checkpoints are still
   available as Kaggle datasets.

## Model export

Per spec, the best-performing checkpoint across the grid should be published as a Kaggle dataset.
Based on the (currently unresolved) numbers above, DINOv3 @ 50% labels has the highest mAP50-95
(0.515) among all newly-run ablation configurations. Confirm this checkpoint is still saved from
`groupj-partb-bonus-dinov3-rho50.ipynb` and publish it as `groupj-partb-bonus-best-checkpoint` if
not already done.